# 6.3 Vectorruimten — hoe een georganiseerde ruimte eruitziet

6.1 projecteerde ruwe *kolommen* — pixels, metingen — naar twee dimensies. 6.2 bouwde met de hand een vector op, één trigram-telling tegelijk. Dit notebook bekijkt een vector die niemand in deze zaal heeft ontworpen: de output van een encoder getraind op enkele miljoenen foto's, eenmalig gepubliceerd en gedownload in plaats van opnieuw berekend.

De vraag is niet "werkt dimensionaliteitsreductie hierop" — 6.1 beantwoordde dat al. De vraag is: **hoe ziet een representatie eruit als deze daadwerkelijk goed is**, en hoe controleer je of een afbeelding ervan je iets echts laat zien. Twee ideeën dragen het antwoord — cosinus-similariteit, en een baseline berekend zónder enig plaatje — en beide komen terug in [06.4](06.4-your-own-vectors%20jeffrey.ipynb), waar de vectoren van jou zijn en de data veel minder meewerkt.

> Vereist `torch` en `vectormesh`: eenmalig lokaal `uv sync --extra huggingface`.

In [ ]:
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

from goad_toolkit.visualizer import PlotSettings, ProjectionPlot
from vectormesh import VectorCache


## Een cache is vectoren plus een manifest

Een afbeelding door een encoder halen is het dure onderdeel van deze pipeline, en het hangt niet af van wat je van plan bent met het resultaat te doen — dus dat wordt één keer gedaan en gepubliceerd. `VectorCache.from_hub` downloadt dat resultaat: een map met vooraf berekende vectoren en een `metadata.json` die ze beschrijft, in plaats van een modeldownload en een GPU.

Dit is hetzelfde idee waarmee de cache-les van de ML-cursus opent; de twee regels hieronder zijn de volledige kosten om bij een echte, fijnmazige, reeds geëncodeerde dataset te komen.

In [ ]:
tr = VectorCache.from_hub("pttrn-io/oxford-flowers-dinov2-small", split="train")
te = VectorCache.from_hub("pttrn-io/oxford-flowers-dinov2-small", split="test")

Xtr, ytr = tr["embed"][:], tr["label"][:]
Xte, yte = te["embed"][:], te["label"][:]
print(f"{len(tr):,} train vectors, {len(te):,} test vectors, {Xtr.shape[1]} dimensions each")
print(f"{te.features['label'].num_classes} flower species")


Geen pixels te bekennen in die map — 102 soorten bloemen, gereduceerd tot een vingerafdruk van 384 getallen per foto door `facebook/dinov2-small`, gedownload in minder dan een seconde. De labels in deze specifieke cache zijn klassenummers in plaats van soortnamen; dat is prima, want niets hieronder hoeft te weten welk nummer een roos is.

## Een georganiseerde ruimte, vóórdat er iets getraind is

Als de encoder zijn werk goed heeft gedaan, zouden foto's van dezelfde soort al dicht bij elkaar moeten liggen — niet omdat er iets is gefit op deze taak, maar omdat "dichtbij" in deze ruimte hoort te betekenen: "lijkt op". Acht soorten, geprojecteerd naar 2D met dezelfde PCA die 6.1 gebruikte, is de directe manier om dat te controleren.

In [ ]:
flower_names = te.features["label"].names
rng = torch.Generator().manual_seed(0)
sample_classes = torch.randperm(len(flower_names), generator=rng)[:8]

mask_te = torch.isin(yte, sample_classes)
coords = PCA(n_components=2).fit_transform(Xte[mask_te].numpy())
labels = np.array([f"class {i}" for i in yte[mask_te].numpy()])

settings = PlotSettings(
    figsize=(7, 6),
    title="Eight flower classes, PCA of raw DINOv2 vectors — nothing trained",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=coords, labels=labels, palette="tab10")


Acht klassen in een handvol hechte groepen, afkomstig van een *lineaire* projectie van ruwe encoder-output — twee of drie soorten overlappen in deze twee componenten, wat is wat twee componenten van de 384 kunnen doen, en de 1-NN check hieronder beslist of ze overlappen in alle 384 dimensies. Vergelijk dit met het MNIST-plaatje uit 6.1: twee componenten van ruwe pixels droegen daar 17% van de variantie en leverden een vlek op. Hier levert dezelfde twee-componenten-PCA duidelijke groepen op, omdat de 384 getallen al per soort georganiseerd waren vóórdat PCA ze ooit aanraakte — de projectie creëert de structuur niet, maar laat je die alleen zien.

## Cosinus-similariteit, fatsoenlijk uitgelegd

Elke afstand die tot nu toe is gebruikt — Manhattan in 6.2, impliciet Euclidisch in elk PCA-plot — behandelt de lengte van een vector als onderdeel van wat het meet. Cosinus-similariteit doet dat niet: het vergelijkt uitsluitend *richting*, wat het de juiste metriek maakt wanneer lengte geen deel uitmaakt van de betekenis.

Een klein voorbeeld maakt het verschil concreet. Drie 2D-vectoren, als stand-in voor drie korte stukjes tekst:

In [ ]:
a = np.array([1.0, 2.0])   # "a message"
b = np.array([3.0, 6.0])    # "a message a message a message" -- same content, said three times
c = np.array([2.0, 1.0])    # a different message entirely


def euclidean(u, v):
    return np.linalg.norm(u - v)


def cosine(u, v):
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))


print(f"euclidean(a, b) = {euclidean(a, b):.2f}   cosine(a, b) = {cosine(a, b):.2f}")
print(f"euclidean(a, c) = {euclidean(a, c):.2f}   cosine(a, c) = {cosine(a, c):.2f}")


Volgens de Euclidische afstand ligt `a` dichter bij `c` — een wezenlijk ander bericht — dan bij `b`, wat `a` drie keer herhaald is en niets nieuws zegt. Volgens cosinus is `b` identiek aan `a` (`1.00`, dezelfde richting) en is `c` duidelijk anders (`0.80`). Cosinus is de metriek die overeenkomt met wat er werkelijk gebeurde: jezelf herhalen is niet iets nieuws zeggen, en lengte zou niet mogen lijken alsof dat wel zo is.

Daarom is cosinus — niet Euclidisch, niet Manhattan — de standaardkeuze voor het vergelijken van zins- en document-embeddings: een bericht van drie woorden en een bericht van drie alinea's kunnen in dezelfde richting wijzen, en cosinus is de enige van de drie die dat ook zou zeggen.

### Maakt het hier uit?

De eigenschap *bijt* pas wanneer lengte varieert om redenen die niets met betekenis te maken hebben — een kort bericht tegenover een lang bericht. Controleer of dat überhaupt waar is voor deze bloemenvectoren vóórdat je aanneemt dat cosinus de alternatieven zal verslaan op deze specifieke dataset.

In [ ]:
norms = Xtr.norm(dim=1)
print(f"vector length: mean {norms.mean():.1f}, std {norms.std():.2f} "
      f"({norms.std() / norms.mean():.1%} of the mean)")


Minder dan 2% variatie. `facebook/dinov2-small` normaliseert intern, dus vrijwel niets van de informatie in deze specifieke vectoren zit in hun lengte — wat betekent dat op *deze* dataset cosinus, Euclidisch en Manhattan het nagenoeg met elkaar eens gaan zijn. Niet omdat de keuze er in het algemeen niet toe doet, maar omdat deze encoder de gelijkmaking die cosinus anders zou moeten doen, al heeft uitgevoerd. [06.4](06.4-your-own-vectors%20jeffrey.ipynb) is een ander verhaal: niemand normaliseert een WhatsApp-bericht voor jou, en een drie woorden tellend "haha" heeft bij lange na niet de lengte van een verhaal van drie alinea's.

## De baseline: 1-nearest-neighbour, helemaal zonder training

Normaliseer, neem een inwendig product (dot product), zoek de dichtstbijzijnde trainingsvector, kopieer het label. Vier regels, geen loss-functie, geen optimizer — en het is het getal dat elke op embeddings gebaseerde methode in deze ruimte moet verslaan.

In [ ]:
def one_nn_accuracy(Xtr, ytr, Xte, yte, p=None):  # noqa: N803
    if p is None:  # cosine: normalise first, then it is just a dot product
        train = torch.nn.functional.normalize(Xtr, dim=1)
        test = torch.nn.functional.normalize(Xte, dim=1)
        nearest = (test @ train.T).argmax(dim=1)
    else:
        nearest = torch.cdist(Xte, Xtr, p=p).argmin(dim=1)
    return (ytr[nearest] == yte).float().mean().item()


print(f"cosine     : {one_nn_accuracy(Xtr, ytr, Xte, yte):.1%}")
print(f"euclidean  : {one_nn_accuracy(Xtr, ytr, Xte, yte, p=2):.1%}")
print(f"manhattan  : {one_nn_accuracy(Xtr, ytr, Xte, yte, p=1):.1%}")


**99,1% op 102 fijnmazige klassen, met niets dat een model genoemd zou kunnen worden** — en de drie metrieken landen binnen twee tienden van een procentpunt van elkaar, precies zoals de lengtecontrole hierboven voorspelde. Dit is de echte taak van de baseline: het beantwoordt de vraag *"is de clustering in het plaatje hierboven echt, of is het iets wat de projectie heeft verzonnen?"* — hier berekend in de oorspronkelijke 384-dimensionale ruimte, niet de 2D-ruimte, dus het verifieert wat dat plaatje slechts suggereerde.

## Controleren zoals 6.1 je heeft geleerd

Eén enkele perplexity, eenmalig, is een demo. De regel van 6.1 was om er meer dan één te draaien en te kijken wat standhoudt — lidmaatschap overleeft meestal, geometrie niet — en om het plaatje te controleren aan de hand van een getal dat berekend is zonder enig plaatje. Dezelfde gewoonte, deze dataset.

In [ ]:
PERPLEXITIES = (5, 30, 100)
mask_tr = torch.isin(ytr, sample_classes)
X = Xtr[mask_tr].numpy()
y = np.array([f"class {i}" for i in ytr[mask_tr].numpy()])

embeddings = {
    p: TSNE(n_components=2, perplexity=p, random_state=42, init="pca").fit_transform(X)
    for p in PERPLEXITIES
}

settings = PlotSettings(
    figsize=(11, 4),
    title="The same eight classes, three perplexities",
    xlabel="",
    ylabel="",
    max_cols=3,
    subplot_titles=[f"perplexity {p}" for p in PERPLEXITIES],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=len(PERPLEXITIES))
for ax, p in zip(axes, PERPLEXITIES):
    host.plot_on_axes(ProjectionPlot(settings), ax, coordinates=embeddings[p],
                      labels=y, palette="tab10", legend=False, s=8)

original_space_silhouette = silhouette_score(X, y)
print(f"\nsilhouette, original 384-dim space: {original_space_silhouette:.3f}")
for p, embedding in embeddings.items():
    print(f"silhouette, t-SNE at perplexity {p:>3d}: {silhouette_score(embedding, y):.3f}")

Lidmaatschap houdt stand bij elke perplexity — de acht klassen blijven acht klassen van 5 tot 100. De 2D-silhouette-scores (0,71–0,77) vallen hoger uit dan die van de 384-dimensionale ruimte (0,41), wat verwacht wordt en geen tegenstrijdigheid is: t-SNE optimaliseert immers exact voor hechte, gescheiden 2D-buurten, dus zijn eigen score is niet direct vergelijkbaar met die van de oorspronkelijke ruimte. Wat *wel* vergelijkbaar is, en wat er echt toe doet, is dat de rangschikking en het verhaal niet veranderen met de knop — wat de controle is waarvan 6.1 een hele sectie lang betoogde dat je die nooit mag overslaan.

## Wat mee te nemen naar 6.4

1. **Een cache is vectoren plus een manifest.** Lees `metadata.json` — of hier: de shapes en het aantal labels — vóórdat je één regel modelcode schrijft.
2. **Cosinus negeert magnitude met opzet.** Controleer of dat doel überhaupt nodig is op jouw data — dat was hier niet zo, omdat de encoder de lengtes al gelijk had getrokken. Het is nodig op het moment dat lengte varieert om redenen die niets met betekenis te maken hebben.
3. **Een baseline berekend zonder plaatje vertelt je of het plaatje echt is.** 99,1% in 384 dimensies is het getal dat de 2D-scatterplot slechts deed vermoeden.

Alles hierboven werkte omdat één rij één bloem was, zuiver gelabeld, één keer geëncodeerd. [06.4](06.4-your-own-vectors%20jeffrey.ipynb) neemt die luxe weg: de vectoren zijn van jou, een rij is een WhatsApp-bericht, en een groot deel daarvan telt vijf woorden of minder. Het eerste probleem daar is niet welke metriek je moet gebruiken — het is wat je überhaupt "één rij" noemt.